In [ ]:
# Colab setup: install packages not preinstalled on Colab (safe to re-run)
!pip install -q abess

# Reproduce-then-Extend — Economic Shocks & Regional Elite Splits (Comparative Political Economy)  ·  **Day 1 tutorial**

> **Published study.** Vall-Prat, P. (2022). "Economic Shocks, Mobilization, and Regional Elite Splits."
> *Comparative Political Studies* 56(2) (doi:10.1177/00104140221089641). Replication data: Harvard Dataverse
> doi:10.7910/DVN/NYS9UR. The paper asks **why regional elites break away and form regionalist parties**,
> using the historical emergence of the Catalan **Lliga** party (1901–1918) across **60 electoral districts**.
> The reported cross-sectional regression has several predictors on a small sample — the setting where
> **regularization and variable selection** earn their keep.

## Background

Why do regional elites split from the national elite and build their own parties? Vall-Prat argues that an
**economic shock** politicizes pre-existing regional grievances. The case is **Catalonia** around 1900: Spain's
loss of its last colonies in **1898** collapsed the protected colonial markets on which Catalan **textile**
manufacturers depended, and this **colonial trade shock** — falling hardest on the industrial districts — pushed
Catalan economic elites toward the regionalist **Lliga Regionalista**. We take the **cross-section of 60
electoral districts** and ask which district characteristics predict Lliga success.

## Data and codebook

**Unit of analysis:** a Catalan **electoral district**; n = 60. Data are the author's replication file.

| Variable | Definition |
|---|---|
| `Lliga` | number of Lliga electoral wins in the district, 1901–1918 (0–12) **(outcome)** |
| `ColShock` | exposure to the **1898 colonial trade shock** (loss of colonial textile markets) |
| `IdMob` | strength of Catalan **identity mobilization** in the district |
| `logPop` | log district population |
| `Manre_dist` | distance to **Manresa** (the industrial town where the 1892 "Bases de Manresa" Catalanist programme was drafted) |
| `dum_Landowners` | 1 if the district was dominated by large **landowners** |
| `BCN` | 1 if the district is in the city of **Barcelona** |
| `ProvId` | province (Barcelona, Girona, Lleida, Tarragona) — enters as **fixed effects** |

## Descriptive results

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV, LinearRegression, lasso_path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
rng = np.random.RandomState(2026)

In [ ]:
d = pd.read_csv('https://raw.githubusercontent.com/desmarais-lab/desmarais-lab.github.io/master/istanbul_bilgi_ml_files/data/catalan_lliga.csv')
outcome = 'Lliga'
subst = ['ColShock','IdMob','logPop','Manre_dist','dum_Landowners','BCN']    # substantive predictors
# province fixed effects (as the paper's ib8.ProvId), then the design matrix
Xdf = pd.concat([d[subst], pd.get_dummies(d['ProvId'].astype('category'), prefix='prov', drop_first=True)], axis=1).astype(float)
preds = list(Xdf.columns)
print(f'{d.shape[0]} districts x {len(preds)} predictors (6 substantive + province FE); p/n = {len(preds)/d.shape[0]:.2f}')
d[[outcome]+subst].describe().T[['mean','std','min','max']].round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].hist(d[outcome], bins=range(0,14), color='#a6cee3', edgecolor='white', align='left')
ax[0].set_title('Outcome: Lliga wins per district'); ax[0].set_xlabel('number of Lliga wins, 1901-1918')
cors = d[subst].corrwith(d[outcome]).sort_values()
ax[1].barh(cors.index, cors.values, color=['#1f78b4' if v>0 else '#e31a1c' for v in cors.values])
ax[1].axvline(0, color='grey'); ax[1].set_title('Correlation with Lliga success'); plt.tight_layout()

## Reproduce the published regression

The reported model regresses Lliga success on the colonial economic shock and district controls, with
**province fixed effects** (Vall-Prat 2022).

In [ ]:
X = Xdf.values.astype(float); y = d[outcome].values.astype(float)
ols = sm.OLS(y, sm.add_constant(X)).fit()
print(f'n = {int(ols.nobs)}, p = {len(preds)}, in-sample R^2 = {ols.rsquared:.3f}')
coef = pd.Series(ols.params[1:], index=preds); pval = pd.Series(ols.pvalues[1:], index=preds)
for v in ['ColShock','IdMob','logPop','dum_Landowners']:
    print(f'   {v:16s} coef = {coef[v]:+.3f}   p = {pval[v]:.3f}')

**Confirmation against the published study.** The model recovers the paper's central result: the
**colonial trade shock is a positive, significant predictor** of Lliga success (`ColShock` $\approx +0.37$,
$p \approx 0.003$) — the districts whose textile economies were most exposed to the 1898 collapse of colonial
markets were most likely to send the regionalist Lliga to parliament, exactly Vall-Prat's argument that an
**economic shock drove the regional elite split**. In-sample $R^2 \approx 0.52$. But nine predictors on sixty
districts is a lot to ask of OLS, and — as we'll see — it over-fits.

## Regularization & variable selection

With **60 districts** and **`r len(preds)` predictors** (p/n $\approx$ 0.15), ordinary least squares has few
observations per coefficient and **over-fits**. Penalized regression — lasso ($L_1$), ridge ($L_2$), elastic
net — shrinks the coefficients to trade a little bias for lower variance.

In [ ]:
Xs = StandardScaler().fit_transform(X)
liCV = LassoCV(cv=5, random_state=0, max_iter=100000).fit(Xs, y)
n_keep = int(np.sum(liCV.coef_ != 0))
print(f'At the CV-optimal penalty the lasso keeps {n_keep} of {len(preds)} predictors (rest set to exactly 0).')
pd.DataFrame({'OLS(std)': LinearRegression().fit(Xs,y).coef_,
              'Lasso': liCV.coef_,
              'Ridge': RidgeCV(alphas=np.logspace(-2,3,40)).fit(Xs,y).coef_,
              'ElasticNet': ElasticNetCV(cv=5, l1_ratio=0.5, random_state=0, max_iter=100000).fit(Xs,y).coef_},
             index=preds).round(3)

**Which fits best out of sample?** With n = 60 a single split is noisy, so we **repeat a 70/30 split 60
times**: each replicate tunes the penalty by cross-validation on the training districts and scores once on the
held-out districts, reporting held-out **$R^2$**.

In [ ]:
res = {k:[] for k in ['OLS','Lasso','Ridge','ElasticNet']}
for r in range(60):
    Xtr,Xte,ytr,yte = train_test_split(X, y, test_size=0.30, random_state=r)
    sc = StandardScaler().fit(Xtr); Ztr, Zte = sc.transform(Xtr), sc.transform(Xte)
    for name, mdl in [('OLS', LinearRegression()),
                      ('Lasso', LassoCV(cv=5, random_state=0, max_iter=100000)),
                      ('Ridge', RidgeCV(alphas=np.logspace(-2,3,40))),
                      ('ElasticNet', ElasticNetCV(cv=5, l1_ratio=0.5, random_state=0, max_iter=100000))]:
        res[name].append(r2_score(yte, mdl.fit(Ztr, ytr).predict(Zte)))
mean = {k: np.mean(v) for k,v in res.items()}
for k,v in mean.items(): print(f'{k:12s} held-out R^2 = {v:+.3f}')
best = max(['Lasso','Ridge','ElasticNet'], key=lambda k: mean[k])
print(f'\nBest penalization: {best}. Held-out R^2 {mean["OLS"]:+.3f} (OLS) -> {mean[best]:+.3f} ({best}), '
      f'a {100*(mean[best]-mean["OLS"])/abs(mean["OLS"]):.0f}% improvement.')

Because there are so few districts per predictor, unpenalized OLS **over-fits** and predicts held-out
districts worse; the penalized models **predict new districts more accurately** (here the best penalty lifts
held-out $R^2$ from about 0.22 to 0.28), while the lasso keeps a compact set of drivers led by the **colonial
trade shock**. It is the same lesson as the other small-sample example (Palazzolo & Moscardelli): with few
observations per predictor, a penalty buys real out-of-sample accuracy.

In [ ]:
alphas, cpath, _ = lasso_path(Xs, y, n_alphas=40)
plt.figure(figsize=(6,3.4)); plt.plot(np.log10(alphas), cpath.T, color='#1f78b4', alpha=.4)
plt.xlabel('log10(alpha)  (more penalty ->)'); plt.ylabel('coefficient'); plt.title('Lasso coefficient paths'); plt.tight_layout()

## Best-subset selection with ABESS

Lasso reaches a sparse model through shrinkage. **Best-subset selection** searches directly for the subset of
predictors that fits best; the **adaptive best-subset (ABESS)** algorithm does so efficiently and chooses the
subset size automatically.

In [ ]:
try:
    from abess.linear import LinearRegression as AbessLR
    ab = AbessLR(support_size=range(1, 8)).fit(Xs, y)
    sel = [p for p,c in zip(preds, ab.coef_) if c!=0]
    print(f'ABESS selects a best subset of {len(sel)} predictors: {sel}')
except Exception as e:
    print('Install abess to run this cell:  !pip install abess')
    print('(', e, ')')

Best subset and the lasso agree on the core of Vall-Prat's story — the **colonial economic shock** and
identity mobilization — reached by direct search rather than shrinkage.

## Takeaway

This example illustrates **regularization in a small-sample design** using a real published study: Vall-Prat
(2022) fits nine predictors to sixty historical Catalan districts, so ordinary regression over-fits and
predicts new districts poorly, while lasso/ridge/elastic-net (`scikit-learn`) and best-subset selection
(`abess`) predict more accurately **and** name the compact set of drivers — with the **1898 colonial trade
shock** at the center — behind the emergence of the regionalist Lliga.

## Recommended exercises

1. Drop the province fixed effects and refit; how do the OLS coefficients and the held-out gain change?
2. Add the raw industry variables (cotton looms, spindles, wool factories) and watch the penalty's edge over OLS grow with the predictor count.
3. Compare `ColShock` in the lasso vs OLS models — how much does the penalty shrink it, and does it survive?
4. Report which predictors ABESS and the lasso agree on, and interpret that compact model substantively.